**IMPORTS:**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.lines import Line2D
from IPython.display import Image, display
import os
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import f1_score
import matplotlib.animation as animation
from IPython.display import HTML, display
os.makedirs("optimizer_outputs", exist_ok=True)

**Question 1:**

In [ ]:
#Loss surfaces given
def surface1(theta):
    x, y = theta
    return x**2 + y**2
def surface2(theta):
    x, y = theta
    return x**2 + 200*y**2
def surface3(theta):
    x, y = theta
    return x**2 - y**2
#gradients of loss functions
def grad1(theta):
    x, y = theta
    return np.array([2*x, 2*y])
def grad2(theta):
    x, y = theta
    return np.array([2*x, 400*y])
def grad3(theta):
    x, y = theta
    return np.array([2*x, -2*y])

In [ ]:
def inside_bounds(theta, xlim, ylim):
    x, y = theta
    return (
        xlim[0] <= x <= xlim[1]
        and
        ylim[0] <= y <= ylim[1]
    )

In [ ]:
#Gradient descent implementation
def gradient_descent(grad_fn, start, eta=0.01, steps=1000, xlim=(-10, 10), ylim=(-10, 10)):
    theta = np.array(start, dtype=float)
    trajectory = [theta.copy()]
    for t in range(steps):
        g = grad_fn(theta)
        while True:
            new_theta = theta - eta * g
            if inside_bounds(new_theta, xlim, ylim):
                break
            eta /= 2 #half the eta to be in bounds
        theta = new_theta
        trajectory.append(theta.copy())
    return np.array(trajectory), eta

In [ ]:
#Momentum implementation
def momentum(grad_fn, start, eta=0.01, beta=0.9, steps=1000, xlim=(-10, 10), ylim=(-10, 10)):
    theta = np.array(start, dtype=float)
    velocity = np.zeros_like(theta)
    trajectory = [theta.copy()]
    for t in range(steps):
        g = grad_fn(theta)
        new_velocity = beta * velocity + g
        while True:
            new_theta = theta - eta * new_velocity
            if inside_bounds(new_theta, xlim, ylim):
                break
            eta /= 2 #half the eta to be in bounds
        velocity = new_velocity
        theta = new_theta
        trajectory.append(theta.copy())
    return np.array(trajectory), eta

In [ ]:
#Stochastic gradient descent(SGD)
def sgd(grad_fn, start, eta=0.01, sigma=0.01, steps=1000, xlim=(-10, 10), ylim=(-10, 10)):
    theta = np.array(start, dtype=float)
    trajectory = [theta.copy()]
    for t in range(steps):
        g = grad_fn(theta)
        noise = sigma * np.random.randn(*theta.shape)
        noisy_grad = g + noise
        while True:
            new_theta = theta - eta * noisy_grad
            if inside_bounds(new_theta, xlim, ylim):
                break
            eta /= 2 #half the eta to be in bounds
        theta = new_theta
        trajectory.append(theta.copy())
    return np.array(trajectory), eta

In [ ]:
#Adam implemntation
def adam(grad_fn, start, eta=0.01, beta1=0.9, beta2=0.999, eps=1e-8, steps=1000, xlim=(-10, 10), ylim=(-10, 10)):
    theta = np.array(start, dtype=float)
    m = np.zeros_like(theta)
    v = np.zeros_like(theta)
    trajectory = [theta.copy()]
    for t in range(1, steps + 1):
        g = grad_fn(theta)
        new_m = beta1 * m + (1 - beta1) * g
        new_v = beta2 * v + (1 - beta2) * (g ** 2)
        m_hat = new_m / (1 - beta1 ** t)
        v_hat = new_v / (1 - beta2 ** t)
        update = m_hat / (np.sqrt(v_hat) + eps)
        while True:
            new_theta = theta - eta * update
            if inside_bounds(new_theta, xlim, ylim):
                break
            eta /= 2 #half the eta to be in bounds
        m = new_m
        v = new_v
        theta = new_theta
        trajectory.append(theta.copy())
    return np.array(trajectory), eta

For SGD and MiniBatch-SGD, noise is added to the gradient since these loss surfaces are mathematical functions and dosn't contain actual training samples. The added Gaussian noise helps to simulate the randomness normally present during stochastic optimization.

For MiniBatch-SGD, a smaller value of sigma = 0.005 is used compared to sigma = 0.01 for SGD. This represents the idea that using a minibatch usually reduces the randomness in the gradient update compared to using a single stochastic gradient.

In [ ]:
#Minibatch-SGD implementation
def minibatch_sgd(grad_fn, start, eta=0.01, sigma=0.005, steps=1000, xlim=(-10, 10), ylim=(-10, 10)):
    theta = np.array(start, dtype=float)
    trajectory = [theta.copy()]
    for t in range(steps):
        g = grad_fn(theta)
        noise = sigma * np.random.randn(*theta.shape)
        noisy_grad = g + noise
        while True:
            new_theta = theta - eta * noisy_grad
            if inside_bounds(new_theta, xlim, ylim):
                break
            eta /= 2 #half the eta to be in bounds
        theta = new_theta
        trajectory.append(theta.copy())
    return np.array(trajectory), eta

In [ ]:
#optimisers used and surfaces
optimizers = {"GD": gradient_descent, "Momentum": momentum, "SGD": sgd, "Minibatch-SGD": minibatch_sgd, "Adam": adam}
surfaces = {

    "Surface 1": {
        "loss": surface1,
        "grad": grad1,
        "start": (-4, 4),

        "xlim": (-5, 5),
        "ylim": (-5, 5)
    },

    "Surface 2": {
        "loss": surface2,
        "grad": grad2,
        "start": (-4, 3),

        "xlim": (-5, 5),
        "ylim": (-4, 4)
    },

    "Surface 3": {
        "loss": surface3,
        "grad": grad3,
        "start": (-1.5, 0.001),

        "xlim": (-5, 5),
        "ylim": (-5, 5)
    }
}
learning_rates = {

    "Surface 1": {
        "GD": 0.1,
        "Momentum": 0.05,
        "SGD": 0.1,
        "Minibatch-SGD": 0.1,
        "Adam": 0.05
    },

    "Surface 2": {
        "GD": 0.001,
        "Momentum": 0.0005,
        "SGD": 0.001,
        "Minibatch-SGD": 0.001,
        "Adam": 0.05
    },

    "Surface 3": {
        "GD": 0.1,
        "Momentum": 0.05,
        "SGD": 0.1,
        "Minibatch-SGD": 0.1,
        "Adam": 0.05
    }
}

In [ ]:
# Computing to find minima on each surface using various optimisers given
MAX_STEPS = 1000
results = {}
for sname, sinfo in surfaces.items():
    print(f"\nRunning {sname}")
    results[sname] = {}
    for oname, opt_fn in optimizers.items():
        print(f"  {oname}")
        eta = learning_rates[sname][oname]
        kwargs = {}
        if oname == "Momentum":
            kwargs = {
                "beta": 0.9
            }
        elif oname == "Adam":
            kwargs = {
                "beta1": 0.9,
                "beta2": 0.999
            }
        elif oname == "SGD":
            kwargs = {
                "sigma": 0.01
            }
        elif oname == "Minibatch-SGD":
            kwargs = {
                "sigma": 0.005
            }
        trajectory, eta_used = opt_fn(
            sinfo["grad"],
            sinfo["start"],
            eta=eta,
            steps=MAX_STEPS,
            xlim=sinfo["xlim"],
            ylim=sinfo["ylim"],
            **kwargs
        )
        results[sname][oname] = (
            trajectory,
            eta_used
        )

In [ ]:
CONV_TOL = 1e-2
#checking for convergence, considering the minima for surface 1 and 2 is at (0,0)
def steps_to_converge(traj, tol=CONV_TOL):
    dist = np.linalg.norm(traj, axis=1)
    below = dist < tol
    for t in range(len(below)):
        if np.all(below[t:]):
            return True, t
    return False, None
ESCAPE_DIST = 3.0
#checking when the optimiser escaped the saddle point, considering the saddle point of the surface 3 is (0,0)
def steps_to_escape(traj, dist_thresh=ESCAPE_DIST):
    dist = np.linalg.norm(traj, axis=1)
    for t in range(len(dist)):
        if dist[t] > dist_thresh:
            return True, t
    return False, None



*   For Surface 1 and Surface 2, the minimum is at (0, 0). Convergence is checked using the distance from (0, 0) with a tolerance of 1e-2. The optimizer is considered converged only if it reaches this tolerance and stays within it for the remaining steps.

* For Surface 3, there is no minimum because it is a saddle surface. Therefore, the number of steps taken to clearly move away from the saddle point (0, 0) is recorded. The starting point is already around 1.5 units away from the centre, so an escape distance of 3.0 is used to represent clear movement away from the saddle.

In [ ]:
#creating table for checking if the optimisers could coverge to find minima or not
rows = []
for sname in surfaces:
    is_saddle = (sname == "Surface 3")
    for oname in optimizers:
        trajectory, eta_used = results[sname][oname]
        if not is_saddle:
            reached, step = steps_to_converge(trajectory)
            rows.append({
                "Surface": sname,
                "Optimizer": oname,
                "Initial eta": learning_rates[sname][oname],
                "Final eta": eta_used,
                "Result": "Reached minimum" if reached else "Did not converge",
                "Steps": step if reached else ">1000"
            })
        else:
            escaped, step = steps_to_escape(trajectory)
            rows.append({
                "Surface": sname,
                "Optimizer": oname,
                "Initial eta": learning_rates[sname][oname],
                "Final eta": eta_used,
                "Result": "Escaped saddle" if escaped else "Did not escape",
                "Steps": step if escaped else ">1000"
            })
table = pd.DataFrame(rows)
table

In [ ]:
#Creating grid points for contour plots
def create_grid(surface_info, resolution=300):

    x = np.linspace(
        surface_info["xlim"][0],
        surface_info["xlim"][1],
        resolution
    )

    y = np.linspace(
        surface_info["ylim"][0],
        surface_info["ylim"][1],
        resolution
    )

    X, Y = np.meshgrid(x, y)

    Z = surface_info["loss"]([X, Y])

    return X, Y, Z


colors = {
    "GD": "blue",
    "Momentum": "orange",
    "SGD": "green",
    "Minibatch-SGD": "red",
    "Adam": "purple"
}


def plot_static_trajectory(sname, sinfo, results):

    X, Y, Z = create_grid(sinfo)

    fig, ax = plt.subplots(figsize=(8, 7))

    # Contour lines
    contour = ax.contour(
        X,
        Y,
        Z,
        levels=25
    )

    ax.clabel(
        contour,
        inline=True,
        fontsize=8
    )

    # Plot optimizer trajectories
    for oname in optimizers:

        trajectory, eta_used = results[sname][oname]

        ax.plot(
            trajectory[:, 0],
            trajectory[:, 1],
            label=oname,
            color=colors[oname],
            linewidth=2
        )

        # Starting point
        ax.scatter(
            trajectory[0, 0],
            trajectory[0, 1],
            color=colors[oname],
            marker="o",
            s=40
        )

        # Final point
        ax.scatter(
            trajectory[-1, 0],
            trajectory[-1, 1],
            color=colors[oname],
            marker="x",
            s=60
        )

    # Origin
    ax.scatter(
        0,
        0,
        color="black",
        marker="*",
        s=120,
        label="(0,0)"
    )

    ax.set_xlim(sinfo["xlim"])
    ax.set_ylim(sinfo["ylim"])

    ax.set_xlabel("x")
    ax.set_ylabel("y")

    ax.set_title(f"{sname} - Optimizer Trajectories")

    ax.legend()

    ax.grid(alpha=0.2)

    plt.tight_layout()

    # Display the plot directly in Colab
    plt.show()

In [ ]:
for sname, sinfo in surfaces.items():
    plot_static_trajectory(sname, sinfo, results)

In [ ]:
# Creating an animated GIF for one surface

def create_optimizer_gif(sname, sinfo, results, fps=20):

    # Create contour plot
    X, Y, Z = create_grid(sinfo)

    fig, ax = plt.subplots(figsize=(8, 7))

    contour = ax.contour(
        X,
        Y,
        Z,
        levels=25
    )

    ax.clabel(
        contour,
        inline=True,
        fontsize=8
    )

    # Find the maximum number of frames
    max_frames = max(
        len(results[sname][oname][0])
        for oname in optimizers
    )

    # Create lines and moving dots for each optimizer
    lines = {}
    dots = {}

    for oname in optimizers:

        line, = ax.plot(
            [],
            [],
            label=oname,
            color=colors[oname],
            linewidth=2
        )

        dot, = ax.plot(
            [],
            [],
            marker="o",
            color=colors[oname],
            markersize=7
        )

        lines[oname] = line
        dots[oname] = dot

    # Mark the reference point
    ax.scatter(
        0,
        0,
        color="black",
        marker="*",
        s=120,
        label="(0,0)"
    )

    ax.set_xlim(sinfo["xlim"])
    ax.set_ylim(sinfo["ylim"])

    ax.set_xlabel("x")
    ax.set_ylabel("y")

    ax.set_title(
        f"{sname} - Optimizer Trajectories"
    )

    ax.legend()

    ax.grid(alpha=0.2)

    # Add step counter
    step_text = ax.text(
        0.02,
        0.95,
        "",
        transform=ax.transAxes,
        fontsize=12,
        verticalalignment="top",
        bbox=dict(
            boxstyle="round",
            facecolor="white",
            alpha=0.8
        )
    )

    # Update optimizer paths and dots for each frame
    def update(frame):

        for oname in optimizers:

            trajectory, eta_used = results[sname][oname]

            current_frame = min(
                frame,
                len(trajectory) - 1
            )

            path = trajectory[:current_frame + 1]

            lines[oname].set_data(
                path[:, 0],
                path[:, 1]
            )

            dots[oname].set_data(
                [trajectory[current_frame, 0]],
                [trajectory[current_frame, 1]]
            )

        step_text.set_text(
            f"Step: {frame}"
        )

        return (
            list(lines.values())
            + list(dots.values())
            + [step_text]
        )

    # Create the animation
    animation = FuncAnimation(
        fig,
        update,
        frames=max_frames,
        interval=1000 / fps,
        blit=True
    )

    # Save the GIF
    filename = f"optimizer_outputs/{sname.replace(' ', '_')}_optimizers.gif"

    animation.save(
        filename,
        writer=PillowWriter(fps=fps)
    )

    plt.close(fig)

    return filename

In [ ]:
#generating gifs
gif_files = {}
for sname, sinfo in surfaces.items():
    print(f"Creating GIF for {sname}...")
    gif_files[sname] = create_optimizer_gif(
        sname,
        sinfo,
        results,
        fps=20
    )
    display(
        Image(
            filename=gif_files[sname]
        )
    )

**QUESTION 2:**

In [ ]:
# Rosenbrock surface
def rosenbrock(theta):
    x, y = theta
    return (1 - x)**2 + 100 * (y - x**2)**2
# Gradient of Rosenbrock
def grad_rosenbrock(theta):
    x, y = theta
    dx = -2 * (1 - x) - 400 * x * (y - x**2)
    dy = 200 * (y - x**2)
    return np.array([dx, dy])

In [ ]:
#surface and optimisers specified
rosenbrock_surface = {
    "loss": rosenbrock,
    "grad": grad_rosenbrock,
    "start": (-1.5, 1.5),

    "xlim": (-2, 2),
    "ylim": (-1, 3)
}
optimizers = {
    "GD": gradient_descent,
    "Momentum": momentum,
    "SGD": sgd,
    "Minibatch-SGD": minibatch_sgd,
    "Adam": adam
}
rosenbrock_learning_rates = {
    "GD": 0.0009,
    "Momentum": 0.001,
    "SGD": 0.0009,
    "Minibatch-SGD": 0.0009,
    "Adam": 0.2
}

In [ ]:
# Running all optimizers on the Rosenbrock surface

MAX_STEPS = 1000
rosenbrock_results = {}
print("\nRunning Rosenbrock")
for oname, opt_fn in optimizers.items():
    print(f"  {oname}")
    eta = rosenbrock_learning_rates[oname]
    kwargs = {}
    if oname == "Momentum":
        kwargs = {
            "beta": 0.9
        }
    elif oname == "Adam":
        kwargs = {
            "beta1": 0.9,
            "beta2": 0.999
        }
    elif oname == "SGD":
        kwargs = {
            "sigma": 0.01
        }
    elif oname == "Minibatch-SGD":
        kwargs = {
            "sigma": 0.005
        }
    trajectory, eta_used = opt_fn(
        rosenbrock_surface["grad"],
        rosenbrock_surface["start"],
        eta=eta,
        steps=MAX_STEPS,
        xlim=rosenbrock_surface["xlim"],
        ylim=rosenbrock_surface["ylim"],
        **kwargs
    )
    rosenbrock_results[oname] = (
        trajectory,
        eta_used
    )
    print(
        f"    Initial eta = {eta}"
    )
    print(
        f"    Final eta   = {eta_used}"
    )
    print(
        f"    Steps       = {len(trajectory)-1}"
    )

In [ ]:
#convergence of rosenbrock
ROSENBROCK_TOL = 1e-2
def steps_to_converge_rosenbrock(traj, tol=ROSENBROCK_TOL):
    #Distance from the true minimum (1,1)
    target = np.array([1.0, 1.0])
    dist = np.linalg.norm(traj - target, axis=1)
    below = dist < tol
    for t in range(len(below)):
        if np.all(below[t:]):
            return True, t
    return False, None

In [ ]:
# Creating results table for Rosenbrock
rows_rosenbrock = []
for oname in optimizers:
    trajectory, eta_used = rosenbrock_results[oname]
    reached, step = steps_to_converge_rosenbrock(
        trajectory
    )
    rows_rosenbrock.append({
        "Surface": "Rosenbrock",
        "Optimizer": oname,
        "Initial eta": rosenbrock_learning_rates[oname],
        "Final eta": eta_used,
        "Result":
            "Reached minimum"
            if reached
            else "Did not converge",
        "Steps":
            step
            if reached
            else ">1000"
    })
rosenbrock_table = pd.DataFrame(
    rows_rosenbrock
)
rosenbrock_table

In [ ]:
# Creating GIF for Rosenbrock
def create_rosenbrock_gif(surface_info, results, fps=20):

    # Create contour plot
    X, Y, Z = create_grid(surface_info)

    fig, ax = plt.subplots(
        figsize=(8, 7)
    )
    levels = np.logspace(
        -2,
        4,
        30
    )
    contour = ax.contour(
        X,
        Y,
        Z,
        levels=levels
    )
    ax.clabel(
        contour,
        inline=True,
        fontsize=8
    )

    # Find the maximum number of frames
    max_frames = max(
        len(results[oname][0])
        for oname in optimizers
    )

    # Create lines and moving dots for each optimizer
    lines = {}
    dots = {}
    for oname in optimizers:
        line, = ax.plot(
            [],
            [],
            label=oname,
            color=colors[oname],
            linewidth=2
        )
        dot, = ax.plot(
            [],
            [],
            marker="o",
            color=colors[oname],
            markersize=7
        )
        lines[oname] = line
        dots[oname] = dot

    # Mark the minimum
    ax.scatter(
        1,
        1,
        color="black",
        marker="*",
        s=150,
        label="Minimum (1,1)"
    )
    ax.set_xlim(
        surface_info["xlim"]
    )
    ax.set_ylim(
        surface_info["ylim"]
    )
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(
        "Rosenbrock - Optimizer Trajectories"
    )
    ax.legend()
    ax.grid(alpha=0.2)
    # Add step counter
    step_text = ax.text(
        0.02,
        0.95,
        "",
        transform=ax.transAxes,
        fontsize=12,
        verticalalignment="top",
        bbox=dict(
            boxstyle="round",
            facecolor="white",
            alpha=0.8
        )
    )

    # Update optimizer paths and dots for each frame
    def update(frame):
        for oname in optimizers:
            trajectory, eta_used = results[oname]
            current_frame = min(
                frame,
                len(trajectory) - 1
            )
            path = trajectory[
                :current_frame + 1
            ]
            lines[oname].set_data(
                path[:, 0],
                path[:, 1]
            )
            dots[oname].set_data(
                [trajectory[current_frame, 0]],
                [trajectory[current_frame, 1]]
            )

        step_text.set_text(
            f"Step: {frame}"
        )
        return (
            list(lines.values())
            + list(dots.values())
            + [step_text]
        )
    # Create the animation
    animation = FuncAnimation(
        fig,
        update,
        frames=max_frames,
        interval=1000 / fps,
        blit=True
    )

    # Save the GIF
    filename = "optimizer_outputs/rosenbrock_optimizers.gif"
    animation.save(
        filename,
        writer=PillowWriter(fps=fps)
    )
    plt.close(fig)
    return filename

In [ ]:
# Generating Rosenbrock GIF
rosenbrock_gif = create_rosenbrock_gif(
    rosenbrock_surface,
    rosenbrock_results,
    fps=20
)
display(
    Image(
        filename=rosenbrock_gif
    )
)

### Observation: Rosenbrock compared with Q1 surfaces

* From the results, the Rosenbrock function was harder to optimize than the surfaces used in Q1. In Q1, the optimizers generally performed well on the bowl surface, while the ravine was more difficult for some methods. All five optimizers were also able to move away from the saddle point. However, for the Rosenbrock surface, GD, Momentum, SGD and Minibatch-SGD did not reach the minimum within 1000 steps. Adam was the only optimizer that reached the minimum, and it took 968 steps.

* The main reason for this difficulty is the shape of the Rosenbrock function:

$$
L(x,y)=(1-x)^2+100(y-x^2)^2
$$

* The minimum is at `(1, 1)`, but it lies inside a narrow and curved valley. The optimizer first has to reach this valley and then slowly move along the curved path towards the minimum. Because the curvature is different in different directions and the direction of the valley also changes, it is difficult to choose one fixed learning rate that works well throughout the optimization.

### Comparison with the Q1 surfaces

Surface 2 from Q1, the ravine,

$$
L(x,y)=x^2+200y^2
$$

also has very different curvature in the `x` and `y` directions. However, its valley is straight, while the Rosenbrock valley is curved. From the results, Momentum was able to converge on Surface 2 in 547 steps and Adam converged in 184 steps. However, on Rosenbrock, Momentum did not converge within 1000 steps, while Adam required 968 steps.

* GD also did not converge within 1000 steps on both Surface 2 and Rosenbrock. SGD and Minibatch-SGD also did not converge on either of these surfaces. The added noise in SGD and Minibatch-SGD makes it harder to move precisely inside the narrow Rosenbrock valley.

* Adam performed the best on Rosenbrock. Its adaptive step sizes help it handle the different gradient scales better than the other optimizers. However, even Adam needed 968 steps, which is much more than the 184 steps on the ravine and 189 steps on the bowl. This shows that the curved shape of the Rosenbrock valley still makes the optimization difficult.

### Comparison of results

| Optimizer | Q1 Surface 1 | Q1 Surface 2 | Q1 Surface 3 | Q2 Rosenbrock |
| --- | ---: | ---: | ---: | ---: |
| GD | 29 | >1000 | Escaped in 44 | >1000 |
| Momentum | 119 | 547 | Escaped in 31 | >1000 |
| SGD | 28 | >1000 | Escaped in 42 | >1000 |
| Minibatch-SGD | 28 | >1000 | Escaped in 49 | >1000 |
| Adam | 189 | 184 | Escaped in 51 | 968 |

* The saddle surface gives a different type of behaviour. All optimizers escaped from it within 31 to 51 steps. However, escaping from a saddle is different from finding a minimum. Once the optimizer moves in the direction of negative curvature, it can move away from the centre quickly.

* Summingup everything we have we have observed, the results show that Rosenbrock was the most difficult surface among the ones tested. Surface 2 already showed that very different curvature in different directions can make optimization difficult. Rosenbrock adds another difficulty because its narrow valley is also curved. Because of this, GD, Momentum, SGD and Minibatch-SGD did not converge within 1000 steps, while Adam was able to reach the minimum but required 968 steps.

**QUESTION 3:**

In [ ]:
np.random.seed(0)
torch.manual_seed(0)

In [ ]:
# Rosenbrock loss function
def rosenbrock(theta):
    x, y = theta
    return (1 - x)**2 + 100 * (y - x**2)**2
START = (-1.5, 1.5)
MAX_STEPS = 1000

In [ ]:
# Rosenbrock function using PyTorch
def rosenbrock_torch(theta):
    x = theta[0]
    y = theta[1]
    return (1 - x)**2 + 100 * (y - x**2)**2

In [ ]:
# Learning rate schedule parameters
ETA0 = 0.1
ETA_MIN = 0.001
STEP_SIZE = 200
GAMMA = 0.995
WARMUP_STEPS = 100
TOTAL_STEPS = 1000

In [ ]:
# Constant learning rate
def constant_schedule(t, eta0=ETA0, **kwargs):
    return eta0
# Step decay learning rate
def step_decay_schedule(
    t,
    eta0=ETA0,
    step_size=STEP_SIZE,
    decay_factor=0.1,
    **kwargs
):
    return eta0 * (decay_factor ** (t // step_size))
# Exponential decay learning rate
def exponential_schedule(
    t,
    eta0=ETA0,
    gamma=GAMMA,
    **kwargs
):
    return eta0 * (gamma ** t)
# Cosine annealing learning rate
def cosine_schedule(
    t,
    eta0=ETA0,
    eta_min=ETA_MIN,
    total_steps=TOTAL_STEPS,
    **kwargs
):
    return (
        eta_min
        + 0.5 * (eta0 - eta_min)
        * (1 + np.cos(np.pi * t / total_steps))
    )
# Linear warmup followed by cosine decay
def warmup_cosine_schedule(
    t,
    eta0=ETA0,
    eta_min=ETA_MIN,
    warmup_steps=WARMUP_STEPS,
    total_steps=TOTAL_STEPS,
    **kwargs
):
    if t <= warmup_steps:
        return eta0 * t / warmup_steps
    else:
        return (
            eta_min
            + 0.5 * (eta0 - eta_min)
            * (
                1
                + np.cos(
                    np.pi
                    * (t - warmup_steps)
                    / (total_steps - warmup_steps)
                )
            )
        )


In [ ]:
# Store the learning rate values for each schedule
schedules = {
    "Constant": constant_schedule,
    "Step Decay": step_decay_schedule,
    "Exponential Decay": exponential_schedule,
    "Cosine Annealing": cosine_schedule,
    "Warmup + Cosine": warmup_cosine_schedule
}

In [ ]:
schedule_values = {}
steps = np.arange(1, TOTAL_STEPS + 1)
for name, schedule_fn in schedules.items():
    schedule_values[name] = np.array([
        schedule_fn(t)
        for t in steps
    ])

In [ ]:
# Plot all learning rate schedules
plt.figure(figsize=(10, 6))
for name, values in schedule_values.items():
    plt.plot(steps, values, label=name)
plt.xlabel("Step")
plt.ylabel("Learning Rate")
plt.title("Learning-Rate Schedules")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Adam implementation with a changing learning rate
def adam_with_schedule(
    schedule_fn,
    start=(-1.5, 1.5),
    steps=1000,
    beta1=0.9,
    beta2=0.999,
    eps=1e-8
):
    # Initialize the starting point
    theta = torch.tensor(
        start,
        dtype=torch.float64,
        requires_grad=True
    )
    # Initialize Adam moment estimates
    m = torch.zeros_like(theta)
    v = torch.zeros_like(theta)
    # Store trajectory, loss and learning rate values
    trajectory = []
    losses = []
    learning_rates = []
    for t in range(1, steps + 1):
        # Store the current position
        trajectory.append(
            theta.detach().numpy().copy()
        )
        # Calculate the current loss and gradient
        loss = rosenbrock_torch(theta)
        losses.append(
            loss.item()
        )
        grad = torch.autograd.grad(
            loss,
            theta
        )[0]
        # Get learning rate for the current step
        eta_t = schedule_fn(t)
        learning_rates.append(eta_t)
        # Update first and second moment estimates
        m = beta1 * m + (1 - beta1) * grad
        v = beta2 * v + (1 - beta2) * (grad ** 2)
        # Bias correction
        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)
        # Update parameters using Adam
        with torch.no_grad():
            theta -= (
                eta_t
                * m_hat
                / (torch.sqrt(v_hat) + eps)
            )
        theta.requires_grad_(True)
    # Store the final position and loss
    trajectory.append(
        theta.detach().numpy().copy()
    )
    losses.append(
        rosenbrock_torch(theta).item()
    )
    return {
        "trajectory": np.array(trajectory),
        "losses": np.array(losses),
        "learning_rates": np.array(learning_rates)
    }

In [ ]:
# Running Adam with all learning-rate schedules
results_q3 = {}
for name, schedule_fn in schedules.items():
    print(f"Running {name}...")
    results_q3[name] = adam_with_schedule(
        schedule_fn,
        start=START,
        steps=MAX_STEPS,
        beta1=0.9,
        beta2=0.999
    )
print("Done!")

In [ ]:
results_q3["Constant"]["trajectory"].shape

In [ ]:
# Creating a summary table for all learning-rate schedules
summary_rows = []
for name, result in results_q3.items():
    trajectory = result["trajectory"]
    losses = result["losses"]
    final_x = trajectory[-1, 0]
    final_y = trajectory[-1, 1]
    final_loss = losses[-1]
    # Find the lowest loss reached during optimization
    best_step = np.argmin(losses)
    best_loss = losses[best_step]
    summary_rows.append({
        "Schedule": name,
        "Final x": final_x,
        "Final y": final_y,
        "Final Loss": final_loss,
        "Best Loss": best_loss,
        "Best Step": best_step
    })
summary_table = pd.DataFrame(summary_rows)
summary_table

In [ ]:
# Finding the schedule with the lowest final loss
best_schedule = summary_table.loc[
    summary_table["Final Loss"].idxmin(),
    "Schedule"
]
best_loss = summary_table["Final Loss"].min()
print("Best schedule:", best_schedule)
print("Final loss:", best_loss)

In [ ]:
# Plotting loss versus step for all schedules
plt.figure(figsize=(10, 6))
for name, result in results_q3.items():

    plt.plot(
        result["losses"],
        label=name
    )
plt.xlabel("Step")
plt.ylabel("Rosenbrock Loss")
plt.title("Loss vs Step on Rosenbrock")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Plotting loss versus step using log scale
plt.figure(figsize=(10, 6))
for name, result in results_q3.items():
    plt.plot(
        result["losses"],
        label=name
    )
plt.yscale("log")
plt.xlabel("Step")
plt.ylabel("Rosenbrock Loss (log scale)")
plt.title("Loss vs Step on Rosenbrock")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Creating grid for Rosenbrock contour plot
x = np.linspace(-2, 2, 500)
y = np.linspace(-1, 3, 500)
X, Y = np.meshgrid(x, y)
Z = (1 - X)**2 + 100 * (Y - X**2)**2

In [ ]:
# Plotting trajectories for different learning-rate schedules
plt.figure(figsize=(10, 7))

levels = np.logspace(
    -2,
    4,
    30
)

plt.contour(
    X,
    Y,
    Z,
    levels=levels
)

for name, result in results_q3.items():

    traj = result["trajectory"]

    plt.plot(
        traj[:, 0],
        traj[:, 1],
        label=name
    )

# Starting point
plt.scatter(
    START[0],
    START[1],
    marker="o",
    s=80,
    label="Start"
)

# True minimum
plt.scatter(
    1,
    1,
    marker="*",
    s=150,
    label="Minimum (1,1)"
)

plt.xlabel("x")
plt.ylabel("y")
plt.title("Adam with Different Learning-Rate Schedules")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Creating GIF for learning-rate schedules on Rosenbrock
def create_q3_gif(
    results,
    filename="q3_rosenbrock_schedules.gif",
    fps=20
):
    # Create Rosenbrock contour grid
    x = np.linspace(-2, 2, 500)
    y = np.linspace(-1, 3, 500)

    X, Y = np.meshgrid(x, y)

    Z = (1 - X)**2 + 100 * (Y - X**2)**2

    levels = np.logspace(
        -2,
        4,
        30
    )
    # Create two panels
    fig, (ax1, ax2) = plt.subplots(
        1,
        2,
        figsize=(14, 6)
    )
    # Rosenbrock contour plot
    ax1.contour(
        X,
        Y,
        Z,
        levels=levels
    )

    ax1.set_xlim(-2, 2)
    ax1.set_ylim(-1, 3)

    ax1.set_xlabel("x")
    ax1.set_ylabel("y")

    ax1.set_title(
        "Adam on Rosenbrock"
    )

    ax1.grid(True)

    steps = np.arange(
        1,
        MAX_STEPS + 1
    )


    for name, result in results.items():

        ax2.plot(
            steps,
            result["learning_rates"],
            label=name
        )

    ax2.set_xlim(
        1,
        MAX_STEPS
    )

    ax2.set_ylim(
        0,
        ETA0 * 1.1
    )

    ax2.set_xlabel("Step")
    ax2.set_ylabel(r"$\eta_t$")

    ax2.set_title(
        "Learning-Rate Schedules"
    )

    ax2.grid(True)

    ax2.legend(
        fontsize=8
    )

    # Store trajectory lines, position dots and learning-rate dots
    lines = {}
    dots = {}
    eta_dots = {}

    for name, result in results.items():

        traj = result["trajectory"]

        # Full trajectory path
        line, = ax1.plot(
            [],
            [],
            label=name
        )

        # Current position
        dot, = ax1.plot(
            [],
            [],
            marker="o",
            linestyle="None",
            markersize=7
        )

        lines[name] = line
        dots[name] = dot

        # Current learning-rate point
        eta_dot, = ax2.plot(
            [],
            [],
            marker="o",
            linestyle="None",
            markersize=6
        )

        eta_dots[name] = eta_dot


    ax1.legend(
        fontsize=8
    )

    # Step counter
    step_text = ax1.text(
        0.02,
        0.95,
        "",
        transform=ax1.transAxes,
        fontsize=12,
        verticalalignment="top"
    )

    # Initialize animation
    def init():

        for name in results:

            lines[name].set_data(
                [],
                []
            )

            dots[name].set_data(
                [],
                []
            )

            eta_dots[name].set_data(
                [],
                []
            )

        step_text.set_text("Step: 0")

        return (
            list(lines.values())
            + list(dots.values())
            + list(eta_dots.values())
            + [step_text]
        )

    # Update each frame
    def update(frame):

        step = min(
            frame,
            MAX_STEPS
        )

        for name, result in results.items():

            traj = result["trajectory"]

            current_traj = traj[:step + 1]

            lines[name].set_data(
                current_traj[:, 0],
                current_traj[:, 1]
            )

            current = traj[step]

            dots[name].set_data(
                [current[0]],
                [current[1]]
            )
            # Get learning rate for the current step
            if step == 0:

                eta = result["learning_rates"][0]

            else:

                eta = result["learning_rates"][
                    min(step - 1, MAX_STEPS - 1)
                ]

            eta_dots[name].set_data(
                [step + 1],
                [eta]
            )

        step_text.set_text(
            f"Step: {step}/{MAX_STEPS}"
        )

        return (
            list(lines.values())
            + list(dots.values())
            + list(eta_dots.values())
            + [step_text]
        )

    # Create animation
    animation = FuncAnimation(
        fig,
        update,
        init_func=init,
        frames=MAX_STEPS + 1,
        interval=1000 / fps,
        blit=True
    )

    # Save GIF
    animation.save(
        filename,
        writer=PillowWriter(fps=fps)
    )

    plt.close(fig)

    print(
        f"GIF saved as: {filename}"
    )

In [ ]:
# Generating and displaying Q3 GIF
create_q3_gif(
    results_q3,
    filename="Q3_Rosenbrock_Learning_Rate_Schedules.gif",
    fps=20
)
display(
    Image(
        filename="Q3_Rosenbrock_Learning_Rate_Schedules.gif"
    )
)

In [ ]:
# Creating GIF for learning-rate schedules
def create_schedule_gif(
    results,
    filename="Q3_learning_rate_schedules.gif",
    fps=20
):

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    steps = np.arange(
        1,
        MAX_STEPS + 1
    )

    lines = {}
    dots = {}
    # Creating lines and moving dots for each schedule
    for name, result in results.items():

        line, = ax.plot(
            [],
            [],
            label=name
        )

        dot, = ax.plot(
            [],
            [],
            marker="o",
            linestyle="None"
        )

        lines[name] = line
        dots[name] = dot

    ax.set_xlim(
        1,
        MAX_STEPS
    )

    ax.set_ylim(
        0,
        ETA0 * 1.1
    )

    ax.set_xlabel("Step")
    ax.set_ylabel(r"$\eta_t$")

    ax.set_title(
        "Learning-Rate Schedules"
    )

    ax.legend()
    ax.grid(True)

    step_text = ax.text(
        0.02,
        0.95,
        "",
        transform=ax.transAxes
    )
    # Initialize animation
    def init():

        for name in results:

            lines[name].set_data(
                [],
                []
            )

            dots[name].set_data(
                [],
                []
            )

        step_text.set_text("Step: 0")

        return (
            list(lines.values())
            + list(dots.values())
            + [step_text]
        )
    # Update animation for each step
    def update(frame):

        step = max(
            1,
            frame
        )

        for name, result in results.items():

            eta_values = result["learning_rates"]

            lines[name].set_data(
                steps[:step],
                eta_values[:step]
            )

            dots[name].set_data(
                [steps[step - 1]],
                [eta_values[step - 1]]
            )

        step_text.set_text(
            f"Step: {step}/{MAX_STEPS}"
        )

        return (
            list(lines.values())
            + list(dots.values())
            + [step_text]
        )

    animation = FuncAnimation(
        fig,
        update,
        init_func=init,
        frames=MAX_STEPS,
        interval=1000 / fps,
        blit=True
    )

    animation.save(
        filename,
        writer=PillowWriter(fps=fps)
    )

    plt.close(fig)

    print(
        f"Saved: {filename}"
    )

In [ ]:
create_schedule_gif(
    results_q3,
    filename="Q3_Learning_Rate_Curves.gif"
)

In [ ]:
display(
    Image(
        filename="Q3_Learning_Rate_Curves.gif"
    )
)

In [ ]:
# Comparing loss during the early optimization steps
EARLY_STEPS = 100
early_rows = []
for name, result in results_q3.items():
    losses = result["losses"]
    early_rows.append({
        "Schedule": name,
        "Loss at step 10": losses[10],
        "Loss at step 50": losses[50],
        "Loss at step 100": losses[100]
    })
early_table = pd.DataFrame(
    early_rows
)
early_table

In [ ]:
constant_early = results_q3[
    "Constant"
]["losses"][EARLY_STEPS]
warmup_early = results_q3[
    "Warmup + Cosine"
]["losses"][EARLY_STEPS]
print(
    "Constant loss at step 100:",
    constant_early
)
print(
    "Warmup + Cosine loss at step 100:",
    warmup_early
)

In [ ]:
final_table = summary_table.copy()
final_table = final_table[
    [
        "Schedule",
        "Final x",
        "Final y",
        "Final Loss",
        "Best Loss",
        "Best Step"
    ]
]
final_table

In [ ]:
# Comparing the trajectories and final positions of all schedules
plt.figure(figsize=(9, 7))

plt.contour(
    X,
    Y,
    Z,
    levels=np.logspace(-2, 4, 30)
)

for name, result in results_q3.items():

    traj = result["trajectory"]

    plt.plot(
        traj[:, 0],
        traj[:, 1],
        label=name
    )

    # final point
    plt.scatter(
        traj[-1, 0],
        traj[-1, 1],
        s=40
    )

plt.scatter(
    START[0],
    START[1],
    marker="o",
    s=100,
    label="Start"
)

plt.scatter(
    1,
    1,
    marker="*",
    s=200,
    label="True minimum"
)

plt.xlabel("x")
plt.ylabel("y")

plt.title(
    "Adam Learning-Rate Schedules on Rosenbrock"
)

plt.legend()
plt.grid(True)
plt.tight_layout()

plt.show()

## Observations

For this experiment, Adam was used on the Rosenbrock function for a maximum of 1000 steps. Five different learning-rate schedules were compared:

* Constant Learning Rate
* Step Decay
* Exponential Decay
* Cosine Annealing
* Linear Warmup + Cosine Decay

For each schedule, the trajectory, loss, and learning rate were recorded at every step.

### Overall Performance

From the results, the Constant Learning Rate performed the best in this experiment. It achieved a final loss of approximately `0.0021` and reached approximately `(0.954, 0.910)`, which is close to the true minimum `(1, 1)`.

The result shows that, for the chosen starting point and the 1000-step limit, keeping the learning rate constant allowed Adam to continue making progress towards the minimum.

### Comparison of the Learning-Rate Schedules

The performance of the schedules based on the final loss was:

* Constant Learning Rate: approximately `0.0021`
* Cosine Annealing: approximately `0.1624`
* Linear Warmup + Cosine Decay: approximately `1.1058`
* Exponential Decay: approximately `2.8483`
* Step Decay: approximately `3.0873`

Cosine annealing was the second-best schedule. It was able to move towards the minimum, but as the learning rate gradually decreased, the optimizer started taking smaller updates. Because of this, it could not get as close to the minimum as the constant learning-rate schedule within 1000 steps.

Step Decay and Exponential Decay performed worse in this experiment. Their learning rates became smaller during optimization, so Adam appeared to slow down before reaching the minimum.

### Did Warmup Improve the Early Steps?

* Based on the results of this experiment, warmup did not improve the overall optimization performance.

* The Linear Warmup + Cosine Decay schedule ended with a final loss of approximately `1.1058`. This was worse than both the Constant Learning Rate and Cosine Annealing schedules.

* During the warmup phase, the learning rate starts small and gradually increases. This can help make optimization more stable in some problems. However, in this experiment, it also meant that Adam made smaller updates during the initial steps. After warmup, the cosine schedule gradually reduced the learning rate again.

* Because of this, the optimizer had less time to make larger updates and move efficiently towards the narrow Rosenbrock valley within the 1000-step limit.

### Why Did the Constant Learning Rate Perform Better?

* The Rosenbrock function has a narrow and curved valley near its minimum. The optimizer needs to make enough progress to enter this valley and then follow it towards `(1, 1)`.

* In this experiment, the constant learning rate allowed Adam to continue taking meaningful steps throughout the optimization. On the other hand, the decay schedules gradually reduced the learning rate, which made the updates smaller before the optimizer could get sufficiently close to the minimum.

* This is why the Constant Learning Rate performed best for the selected learning rate and the 1000-step limit.

### Final Observation

Overall, the experiment shows that the choice of learning-rate schedule can significantly affect Adam's performance on the Rosenbrock surface.

* The Constant Learning Rate performed the best and achieved the lowest final loss.
* Cosine Annealing showed the second-best performance.
* Linear Warmup + Cosine Decay did not improve the results in this experiment.
* Step Decay and Exponential Decay reduced the learning rate too much, causing slower progress towards the minimum.

Therefore, for this particular starting point, learning-rate settings, and 1000-step limit, keeping the learning rate constant was more effective than gradually reducing it.

**QUESTION 4:**

In [ ]:
# Set seed for reproducibility
torch.manual_seed(42)
# Use GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Using device:", device)

In [ ]:
# Convert images to PyTorch tensors
transform = transforms.ToTensor()
# Load training dataset
train_dataset = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)
# Load test dataset
test_dataset = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)
print("Total training images:", len(train_dataset))
print("Total test images:", len(test_dataset))

In [ ]:
# Select a subset of 5000 training images
torch.manual_seed(42)
train_indices = torch.randperm(
    len(train_dataset)
)[:5000]

test_indices = torch.randperm(
    len(test_dataset)
)[:1000]
train_subset = Subset(
    train_dataset,
    train_indices
)
test_subset = Subset(
    test_dataset,
    test_indices
)
print("Training subset size:", len(train_subset))
print("Test subset size:", len(test_subset))

In [ ]:
# Batch size
batch_size = 64
# Create training DataLoader
train_loader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=True
)
# Create test DataLoader
test_loader = DataLoader(
    test_subset,
    batch_size=batch_size,
    shuffle=False
)
print("Training batches:", len(train_loader))
print("Test batches:", len(test_loader))

In [ ]:
from collections import Counter

# Get labels from the training subset
train_labels = [
    train_dataset.targets[i].item()
    for i in train_indices
]
# Get labels from the test subset
test_labels = [
    test_dataset.targets[i].item()
    for i in test_indices
]
# Count samples in each class
train_counts = Counter(train_labels)
test_counts = Counter(test_labels)
print("Training subset class distribution:")
for class_id in range(10):
    print(
        f"Class {class_id}: "
        f"{train_counts[class_id]}"
    )
print("\nTest subset class distribution:")
for class_id in range(10):
    print(
        f"Class {class_id}: "
        f"{test_counts[class_id]}"
    )

In [ ]:
class FashionMLP(nn.Module):
    def __init__(self, dropout_prob=0.0):
        super().__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.network(x)
        return x

In [ ]:
def evaluate_model(model, data_loader, criterion):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0
    all_predictions = []
    all_labels = []
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)
            # Forward pass
            outputs = model(images)
            # Calculate loss
            loss = criterion(
                outputs,
                labels
            )
            # Add loss weighted by batch size
            total_loss += (
                loss.item() * labels.size(0)
            )
            predictions = torch.argmax(
                outputs,
                dim=1
            )
            correct += (
                predictions == labels
            ).sum().item()
            total += labels.size(0)
            all_predictions.extend(
                predictions.cpu().numpy()
            )
            all_labels.extend(
                labels.cpu().numpy()
            )
    # Average loss
    average_loss = total_loss / total
    # Accuracy
    accuracy = correct / total
    # Weighted F1 score
    f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )
    return average_loss, accuracy, f1

In [ ]:
def train_model(
    dropout_prob=0.0,
    num_epochs=50,
    learning_rate=0.001
):
    # Create model
    model = FashionMLP(
        dropout_prob=dropout_prob
    ).to(device)
    # Loss function
    criterion = nn.CrossEntropyLoss()
    # Adam optimizer
    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate
    )
    # Store loss results
    train_loss_history = []
    test_loss_history = []
    # Store accuracy results
    train_accuracy_history = []
    test_accuracy_history = []
    # Store F1 results
    train_f1_history = []
    test_f1_history = []
    # Training loop
    for epoch in range(num_epochs):
        model.train()
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            # Clear previous gradients
            optimizer.zero_grad()
            # Forward pass
            outputs = model(images)
            # Calculate loss
            loss = criterion(
                outputs,
                labels
            )
            # Backpropagation
            loss.backward()
            # Update model parameters
            optimizer.step()
        # Calculate training metrics
        train_loss, train_accuracy, train_f1 = evaluate_model(
            model,
            train_loader,
            criterion
        )
        # Calculate test metrics
        test_loss, test_accuracy, test_f1 = evaluate_model(
            model,
            test_loader,
            criterion
        )
        # Save loss
        train_loss_history.append(train_loss)
        test_loss_history.append(test_loss)
        # Save accuracy
        train_accuracy_history.append(train_accuracy)
        test_accuracy_history.append(test_accuracy)
        # Save F1
        train_f1_history.append(train_f1)
        test_f1_history.append(test_f1)
        # Print results for every epoch
        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Test Loss: {test_loss:.4f} | "
            f"Train Accuracy: {train_accuracy:.4f} | "
            f"Test Accuracy: {test_accuracy:.4f} | "
            f"Train F1: {train_f1:.4f} | "
            f"Test F1: {test_f1:.4f}"
        )
    return {
        "model": model,
        "train_loss": train_loss_history,
        "test_loss": test_loss_history,
        "train_accuracy": train_accuracy_history,
        "test_accuracy": test_accuracy_history,
        "train_f1": train_f1_history,
        "test_f1": test_f1_history
    }

In [ ]:
plain_results = train_model(
    dropout_prob=0.0,
    num_epochs=50
)

In [ ]:
# Epoch numbers
epochs = range(1, 51)
# Create figure
plt.figure(figsize=(12, 5))
# Plot accuracy
plt.subplot(1, 2, 1)
plt.plot(
    epochs,
    plain_results["train_accuracy"],
    label="Training Accuracy"
)
plt.plot(
    epochs,
    plain_results["test_accuracy"],
    label="Test Accuracy"
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Plain MLP Accuracy")
plt.legend()
# Plot F1 score
plt.subplot(1, 2, 2)
plt.plot(
    epochs,
    plain_results["train_f1"],
    label="Training F1"
)
plt.plot(
    epochs,
    plain_results["test_f1"],
    label="Test F1"
)
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.title("Plain MLP F1 Score")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Dropout probabilities required in the question
dropout_probabilities = [0.1, 0.25, 0.5, 0.75]
# Dictionary to store results
dropout_results = {}
# Train a model for each dropout probability
for dropout_prob in dropout_probabilities:
    print("\n" + "=" * 60)
    print(f"Training with Dropout Probability = {dropout_prob}")
    print("=" * 60)
    # Train model
    results = train_model(
        dropout_prob=dropout_prob,
        num_epochs=50
    )
    # Store results
    dropout_results[dropout_prob] = results

In [ ]:
# Epoch numbers
epochs = range(1, 51)
plt.figure(figsize=(10, 6))
for dropout_prob in dropout_probabilities:
    # Training loss
    plt.plot(
        epochs,
        dropout_results[dropout_prob]["train_loss"],
        label=f"Train, Dropout={dropout_prob}",
        linestyle="--"
    )
    # Test loss
    plt.plot(
        epochs,
        dropout_results[dropout_prob]["test_loss"],
        label=f"Test, Dropout={dropout_prob}"
    )
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(
    "Training and Test Loss for Different Dropout Probabilities"
)
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Epoch numbers
epochs = range(1, 51)
plt.figure(figsize=(10, 6))
for dropout_prob in dropout_probabilities:
    plt.plot(
        epochs,
        dropout_results[dropout_prob]["train_accuracy"],
        label=f"Train, Dropout={dropout_prob}",
        linestyle="--"
    )
    plt.plot(
        epochs,
        dropout_results[dropout_prob]["test_accuracy"],
        label=f"Test, Dropout={dropout_prob}"
    )
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Test Accuracy for Different Dropout Probabilities")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Epoch numbers
epochs = range(1, 51)
plt.figure(figsize=(10, 6))
for dropout_prob in dropout_probabilities:
    plt.plot(
        epochs,
        dropout_results[dropout_prob]["train_f1"],
        label=f"Train, Dropout={dropout_prob}",
        linestyle="--"
    )
    plt.plot(
        epochs,
        dropout_results[dropout_prob]["test_f1"],
        label=f"Test, Dropout={dropout_prob}"
    )
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.title("Training and Test F1 Scores for Different Dropout Probabilities")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Accuracy gap between training and test performance
epochs = range(1, 51)
plt.figure(figsize=(10, 6))
for dropout_prob in dropout_probabilities:
    accuracy_gap = []
    for train_acc, test_acc in zip(
        dropout_results[dropout_prob]["train_accuracy"],
        dropout_results[dropout_prob]["test_accuracy"]
    ):
        accuracy_gap.append(
            train_acc - test_acc
        )
    plt.plot(
        epochs,
        accuracy_gap,
        label=f"Dropout={dropout_prob}"
    )
plt.xlabel("Epoch")
plt.ylabel("Training Accuracy - Test Accuracy")
plt.title("Accuracy Gap for Different Dropout Probabilities")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Epoch numbers
epochs = range(1, 51)
plt.figure(figsize=(10, 6))
for dropout_prob in dropout_probabilities:
    # F1 gap between training and test
    f1_gap = [
        train - test
        for train, test in zip(
            dropout_results[dropout_prob]["train_f1"],
            dropout_results[dropout_prob]["test_f1"]
        )
    ]
    plt.plot(
        epochs,
        f1_gap,
        label=f"Dropout={dropout_prob}"
    )
plt.xlabel("Epoch")
plt.ylabel("Training F1 - Test F1")
plt.title(
    "F1 Gap Between Training and Test Performance"
)
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Epoch numbers
epochs = range(1, 51)
plt.figure(figsize=(10, 6))
for dropout_prob in dropout_probabilities:
    # Loss gap between test and training
    loss_gap = [
        test - train
        for train, test in zip(
            dropout_results[dropout_prob]["train_loss"],
            dropout_results[dropout_prob]["test_loss"]
        )
    ]
    plt.plot(
        epochs,
        loss_gap,
        label=f"Dropout={dropout_prob}"
    )
plt.xlabel("Epoch")
plt.ylabel("Test Loss - Training Loss")
plt.title(
    "Loss Gap Between Training and Test Performance"
)
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Epoch numbers
epochs = range(1, 51)
plt.figure(figsize=(7, 5))
# Training loss
plt.plot(
    epochs,
    plain_results["train_loss"],
    label="Training Loss"
)
# Test loss
plt.plot(
    epochs,
    plain_results["test_loss"],
    label="Test Loss"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Plain MLP Training and Test Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Compare the best test performance for each dropout probability
for dropout_prob in dropout_probabilities:
    results = dropout_results[dropout_prob]
    # Lowest test loss
    best_test_loss = min(
        results["test_loss"]
    )
    # Highest test accuracy
    best_test_accuracy = max(
        results["test_accuracy"]
    )
    # Highest test F1 score
    best_test_f1 = max(
        results["test_f1"]
    )
    # Find epochs
    best_loss_epoch = (
        results["test_loss"].index(best_test_loss) + 1
    )
    best_accuracy_epoch = (
        results["test_accuracy"].index(best_test_accuracy) + 1
    )
    best_f1_epoch = (
        results["test_f1"].index(best_test_f1) + 1
    )
    print(f"\nDropout = {dropout_prob}")
    print(
        f"Lowest Test Loss: "
        f"{best_test_loss:.4f} "
        f"(Epoch {best_loss_epoch})"
    )
    print(
        f"Best Test Accuracy: "
        f"{best_test_accuracy:.4f} "
        f"(Epoch {best_accuracy_epoch})"
    )
    print(
        f"Best Test F1 Score: "
        f"{best_test_f1:.4f} "
        f"(Epoch {best_f1_epoch})"
    )

In [ ]:
# Find the dropout probability with the highest best Test F1 score
best_dropout = None
best_f1 = -1
for dropout_prob in dropout_probabilities:
    current_best_f1 = max(
        dropout_results[dropout_prob]["test_f1"]
    )
    if current_best_f1 > best_f1:
        best_f1 = current_best_f1
        best_dropout = dropout_prob
# Get all results for the best dropout
best_results = dropout_results[best_dropout]
# Find best values
best_test_loss = min(
    best_results["test_loss"]
)
best_test_accuracy = max(
    best_results["test_accuracy"]
)
best_test_f1 = max(
    best_results["test_f1"]
)
# Find corresponding epochs
best_loss_epoch = (
    best_results["test_loss"].index(best_test_loss) + 1
)
best_accuracy_epoch = (
    best_results["test_accuracy"].index(
        best_test_accuracy
    ) + 1
)
best_f1_epoch = (
    best_results["test_f1"].index(
        best_test_f1
    ) + 1
)
# Print final conclusion
print("=" * 60)
print("BEST DROPOUT CONFIGURATION")
print("=" * 60)
print(f"\nBest Dropout Probability: {best_dropout}")
print(
    f"\nLowest Test Loss: "
    f"{best_test_loss:.4f} "
    f"(Epoch {best_loss_epoch})"
)
print(
    f"Best Test Accuracy: "
    f"{best_test_accuracy:.4f} "
    f"(Epoch {best_accuracy_epoch})"
)
print(
    f"Best Test F1 Score: "
    f"{best_test_f1:.4f} "
    f"(Epoch {best_f1_epoch})"
)

We trained the same MLP for 50 epochs using 5,000 training images and 1,000 test images, and compared a model with no dropout against models with dropout probabilities of 0.1, 0.25, 0.5, and 0.75. We evaluated each model using training and test loss, accuracy, and macro-F1 score.

*   Dropout = 0.25 gave the best overall performance in our experiment, achieving the lowest test loss and the highest test accuracy and macro-F1 score. This indicates that it provided the best balance between fitting the training data and generalizing to unseen data.

*   As dropout increases, the model becomes more strongly regularized. This makes it harder for the network to memorize the training data, so training performance generally decreases while the difference between training and test performance becomes smaller. This reduction in the train-test gap indicates less overfitting.

*  However, increasing dropout indefinitely is not beneficial. While moderate
dropout can reduce overfitting, very high dropout can make the model too constrained, causing it to underfit. This can result in both training and test performance becoming worse, as seen with the higher dropout settings.


* Overall, the experiment demonstrates the bias-variance tradeoff: no or low dropout can lead to overfitting due to high variance, moderate dropout can improve generalization by reducing variance, while excessive dropout increases bias and can lead to underfitting. Thus, an intermediate dropout probability such as 0.25 provided the best tradeoff for this MLP and dataset.

**Question 5:**

**Parameters and Creating the Room Grid:**

In [ ]:
# Constants from question
sigma = 0.25
rho = 0.20
lambda_repulsion = 1.0
# Room grid size
grid_size = 100
# Creating points from 0 to 1 for x and y axes
x = torch.linspace(0, 1, grid_size)
y = torch.linspace(0, 1, grid_size)
# Creating 100 x 100 grid
X, Y = torch.meshgrid(x, y, indexing="ij")
# Converting grid to list of (x, y) points
grid_points = torch.stack([X.flatten(), Y.flatten()], dim=1)

print("Number of grid points:", len(grid_points))
print("Shape:", grid_points.shape)

**Brightness Calculation and Repulsion Calculation:**

In [ ]:
def calculate_brightness(lamp_positions, grid_points, sigma):
    # squared distance between every grid point and every lamp
    distance_squared = (
        (grid_points.unsqueeze(1) - lamp_positions.unsqueeze(0)) ** 2
    ).sum(dim=2)

    # brightness using the Gaussian formula
    brightness = torch.exp(
        -distance_squared / (2 * sigma ** 2)
    )

    # Add brightness from all lamps at every grid point
    total_brightness = brightness.sum(dim=1)

    return total_brightness

def calculate_repulsion(lamp_positions, rho):

    repulsion = 0
    # Check every pair of lamps
    for i in range(len(lamp_positions)):
        for j in range(i + 1, len(lamp_positions)):

            # Squared distance between two lamps
            distance_squared = (
                (lamp_positions[i] - lamp_positions[j]) ** 2
            ).sum()

            # Add Gaussian repulsion penalty
            repulsion += torch.exp(
                -distance_squared / (2 * rho ** 2)
            )

    return repulsion

**Objective Function:**

In [ ]:
def objective_function(lamp_positions, grid_points, sigma, rho, lambda_repulsion):
    # Calculate total brightness at all room grid points
    total_brightness = calculate_brightness(
        lamp_positions,
        grid_points,
        sigma
    )

    average_brightness = total_brightness.mean()
    # Calculate penalty for lamps that are too close
    repulsion = calculate_repulsion(
        lamp_positions,
        rho
    )

    objective = average_brightness - lambda_repulsion * repulsion

    return objective

**Reusable Optimization Function:**

In [ ]:
# Function to optimize any number of lamps

def optimize_lamps(
    num_lamps=3,
    num_steps=500,
    learning_rate=0.01,
    seed=42
):

    # Set seed for reproducible random starting positions
    torch.manual_seed(seed)

    # Randomly initialize lamp positions inside the room
    lamp_positions = torch.rand(
        num_lamps,
        2,
        requires_grad=True
    )

    # Calculate objective before optimization
    initial_objective = objective_function(
        lamp_positions,
        grid_points,
        sigma,
        rho,
        lambda_repulsion
    ).item()

    # Use Adam optimizer
    optimizer = torch.optim.Adam(
        [lamp_positions],
        lr=learning_rate
    )

    # Store initial positions
    position_history = [
        lamp_positions.detach().clone()
    ]

    # Store initial objective
    objective_history = [
        initial_objective
    ]

    # Optimization loop
    for step in range(num_steps):

        # Clear previous gradients
        optimizer.zero_grad()

        # Calculate objective
        objective = objective_function(
            lamp_positions,
            grid_points,
            sigma,
            rho,
            lambda_repulsion
        )

        # Minimize -J to maximize J
        loss = -objective

        # Calculate gradients
        loss.backward()

        # Update lamp positions
        optimizer.step()

        # Keep lamps inside the unit square
        with torch.no_grad():
            lamp_positions.clamp_(0, 1)

        # Calculate objective AFTER updating positions
        updated_objective = objective_function(
            lamp_positions,
            grid_points,
            sigma,
            rho,
            lambda_repulsion
        )

        # Save current lamp positions
        position_history.append(
            lamp_positions.detach().clone()
        )

        # Save objective corresponding to current positions
        objective_history.append(
            updated_objective.item()
        )

    return (
        lamp_positions.detach(),
        position_history,
        objective_history,
        initial_objective
    )

In [ ]:
# Test the reusable function with 3 lamps

final_positions, position_history, objective_history, initial_objective = optimize_lamps()
print("Initial objective value:", initial_objective)

print("\nFinal objective value:")
print(objective_history[-1])

print("\nFinal lamp positions:")
print(final_positions)

print("\nNumber of saved steps:")
print(len(position_history))

### Note on Negative Initial Objective Values

Some initial objective values are negative in cases like $n > 4$ lamps.

This is expected because the objective is calculated as:

$
J = \text{Average Brightness} - \lambda \times \text{Repulsion}
$

As the lamp positions are initialized randomly, some lamps may initially be close to each other. This creates a high repulsion penalty, which can be larger than the average brightness and result in a negative objective value.

During optimization, the lamps move to improve room coverage and reduce the repulsion penalty. Therefore, the objective value increases to a higher positive value.

**Brightness Map Before and After Optimization:**

In [ ]:
# Get initial lamp positions from optimization history
initial_positions = position_history[0]

# Calculate brightness before optimization
initial_brightness = calculate_brightness(
    initial_positions,
    grid_points,
    sigma
)

# Calculate brightness after optimization
final_brightness = calculate_brightness(
    final_positions,
    grid_points,
    sigma
)

# Use the same brightness scale for both plots
vmax = max(
    initial_brightness.max().item(),
    final_brightness.max().item()
)

# Reshape initial brightness into room grid
initial_brightness_map = initial_brightness.reshape(
    grid_size,
    grid_size
)

print("Initial lamp positions:")
print(initial_positions)

In [ ]:
# Brightness map before optimization
plt.figure(figsize=(7, 5))

plt.imshow(
    initial_brightness_map.detach().numpy(),
    extent=[0, 1, 0, 1],
    origin="lower",
    vmin=0,
    vmax=vmax
)

# Show initial lamp positions
plt.scatter(
    initial_positions[:, 0].detach().numpy(),
    initial_positions[:, 1].detach().numpy(),
    marker="x",
    s=100,
    color="red",
    label="Initial lamps"
)

plt.colorbar(label="Brightness")

plt.xlabel("x")
plt.ylabel("y")
plt.title("Brightness Map Before Optimization")

plt.legend(
    loc="upper left",
    bbox_to_anchor=(-0.35, 1)
)

plt.show()



**Brightness Map After Optimization:**

In [ ]:
# Calculate brightness after optimization

final_brightness = calculate_brightness(
    final_positions,
    grid_points,
    sigma
)

# Reshape into the room grid
final_brightness_map = final_brightness.reshape(
    grid_size,
    grid_size
)

In [ ]:
# Brightness map after optimization
plt.figure(figsize=(7, 5))

plt.imshow(
    final_brightness_map.detach().numpy(),
    extent=[0, 1, 0, 1],
    origin="lower",
    vmin=0,
    vmax=vmax
)

# Show final lamp positions
plt.scatter(
    final_positions[:, 0].detach().numpy(),
    final_positions[:, 1].detach().numpy(),
    marker="x",
    s=100,
    color="red",
    label="Final lamps"
)

plt.colorbar(label="Brightness")

plt.xlabel("x")
plt.ylabel("y")
plt.title("Brightness Map After Optimization")

plt.legend(
    loc="upper left",
    bbox_to_anchor=(-0.35, 1)
)

plt.show()

**Animation of Lamp Movement:**

In [ ]:
# Create figure
fig, ax = plt.subplots(figsize=(6, 6))

# Set room boundaries
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Lamps Changes During Optimization")

# Calculate brightness for the first frame
initial_positions = position_history[0]

initial_brightness = calculate_brightness(
    initial_positions,
    grid_points,
    sigma
)

initial_brightness_map = initial_brightness.reshape(
    grid_size,
    grid_size
)

# Calculate brightness for the final frame
final_positions = position_history[-1]

final_brightness = calculate_brightness(
    final_positions,
    grid_points,
    sigma
)

# Use the same brightness scale throughout the animation
vmax = max(
    initial_brightness.max().item(),
    final_brightness.max().item()
)

# Show brightness map
image = ax.imshow(
    initial_brightness_map.detach().numpy(),
    extent=[0, 1, 0, 1],
    origin="lower",
    animated=True,
    vmin=0,
    vmax=vmax
)

# Add colorbar
plt.colorbar(
    image,
    ax=ax,
    label="Brightness"
)

# Create lamp markers
scatter = ax.scatter(
    [],
    [],
    color="red",
    marker="x",
    s=100,
    label="Lamps"
)

# Animation update function
def update(frame):

    # Get lamp positions at this step
    positions = position_history[frame]

    # Calculate brightness for current lamp positions
    brightness = calculate_brightness(
        positions,
        grid_points,
        sigma
    )

    # Convert brightness into room grid
    brightness_map = brightness.reshape(
        grid_size,
        grid_size
    )

    # Update brightness map
    image.set_data(
        brightness_map.detach().numpy()
    )

    # Update lamp positions
    scatter.set_offsets(
        positions.detach().numpy()
    )

    return image, scatter


# Create animation
anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(position_history),
    interval=50,
    blit=True,
    repeat=True
)

plt.close()

# Increase animation embedding limit
plt.rcParams["animation.embed_limit"] = 50

# Display animation with replay controls
display(HTML(anim.to_jshtml()))

**Effect of Different Random Seeds:**

In [ ]:
# Run optimization with different random seeds

seeds = [1, 10, 42, 100, 200]

results = []

for seed in seeds:

    final_positions_seed, _, objective_history_seed, initial_objective_seed = optimize_lamps(
        num_lamps=3,
        seed=seed
    )

    final_objective = objective_history_seed[-1]

    results.append({
        "seed": seed,
        "initial_objective": initial_objective_seed,
        "final_objective": final_objective,
        "final_positions": final_positions_seed
    })

    print(f"\nSeed: {seed}")
    print("Initial objective:", round(initial_objective_seed, 4))
    print("Final objective:", round(final_objective, 4))
    print("Final positions:")
    print(final_positions_seed)

### Observation Across Random Seeds

Five different random seeds were tested. Although the lamps started from different random positions, all runs reached approximately the same final objective value of 0.7959.

The exact coordinates of the final lamps are different because the lamp positions can be arranged in different orders or symmetric configurations. However, the final placements have a similar spread across the room and give the same objective value.

This suggests that, for the tested seeds, the optimizer consistently found equivalent good solutions.

**Animations for 4, 6 and 8 Lamps:**

In [ ]:
def animate_lamps(num_lamps, seed=42):

    # Run optimization
    final_positions, position_history, objective_history, initial_objective = optimize_lamps(
        num_lamps=num_lamps,
        seed=seed
    )

    # Calculate brightness for the first frame
    initial_positions = position_history[0]

    initial_brightness = calculate_brightness(
        initial_positions,
        grid_points,
        sigma
    )

    # Calculate brightness for the final frame
    final_brightness = calculate_brightness(
        final_positions,
        grid_points,
        sigma
    )

    # Use the same brightness scale throughout the animation
    vmax = max(
        initial_brightness.max().item(),
        final_brightness.max().item()
    )

    # Create figure
    fig, ax = plt.subplots(figsize=(6, 6))

    # Set room boundaries
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(
        f"{num_lamps} Lamps Changes During Optimization"
    )

    # Convert initial brightness into room grid
    initial_brightness_map = initial_brightness.reshape(
        grid_size,
        grid_size
    )

    # Show initial brightness map
    image = ax.imshow(
        initial_brightness_map.detach().numpy(),
        extent=[0, 1, 0, 1],
        origin="lower",
        animated=True,
        vmin=0,
        vmax=vmax
    )

    # Add colorbar
    plt.colorbar(
        image,
        ax=ax,
        label="Brightness"
    )

    # Create lamp markers
    scatter = ax.scatter(
        [],
        [],
        color="red",
        marker="x",
        s=100,
        label="Lamps"
    )

    # Animation update function
    def update(frame):

        # Get lamp positions at this step
        positions = position_history[frame]

        # Calculate brightness for current lamp positions
        brightness = calculate_brightness(
            positions,
            grid_points,
            sigma
        )

        # Convert brightness into room grid
        brightness_map = brightness.reshape(
            grid_size,
            grid_size
        )

        # Update brightness map
        image.set_data(
            brightness_map.detach().numpy()
        )

        # Update lamp positions
        scatter.set_offsets(
            positions.detach().numpy()
        )

        return image, scatter

    # Create animation
    anim = animation.FuncAnimation(
        fig,
        update,
        frames=len(position_history),
        interval=50,
        blit=True,
        repeat=True
    )

    # Close the figure
    plt.close()

    # Increase animation embedding limit from 20 MB to 50 MB
    plt.rcParams["animation.embed_limit"] = 50

    # Return animation with replay controls
    return HTML(anim.to_jshtml())

In [ ]:
display(animate_lamps(4))

In [ ]:
display(animate_lamps(6))

In [ ]:
display(animate_lamps(8))

### Brightness Scale and GIF Interpretation

The colour of the brightness map represents numerical brightness, so the colour scale is important when comparing the results.

A fixed scale using `vmin=0` and `vmax` is used throughout each GIF. Therefore, within the same GIF, the same colour represents the same brightness value from the beginning to the end. This makes the brightness changes during optimization easier to compare correctly.

However, the GIFs for **3, 4, 6, and 8 lamps** are generated separately, and each one may have a different `vmax`. Therefore, the same colour, such as yellow, may represent different brightness values in different GIFs.

Thus, colours should be compared directly **within the same GIF**, but comparisons between GIFs with different numbers of lamps should also consider the actual brightness or objective values. More lamps can also produce higher total brightness because the brightness contributions from all lamps are added together.

### Final Observations and Conclusion

* An interesting observation from the GIFs was that the final pattern changed as the number of lamps increased. For 3 lamps, they formed a roughly triangular arrangement. As the number of lamps increased, the arrangement became more structured. For 6 and 8 lamps, one lamp was often near the centre, while the remaining lamps formed approximately symmetric patterns around it.

* Another observation was that the lamps did not simply move away from each other. They also moved to cover different parts of the room. Therefore, the final positions were influenced by both brightness coverage and repulsion between lamps.

* Overall, different geometric patterns appeared automatically using the same objective function. This was interesting because no specific shape or final arrangement was manually given to the optimizer. The optimizer was only trying to maximize the objective, and the different patterns emerged naturally during optimization.